# exp095_prefix_u_line_residual_target train

Train-side ablation of prefix de-trended U-space residual targets on the fixed exp073 full replay LightGBM feature surface.

## Contents

1. Setup and configuration
2. Exp072 full replay cache and prefix anchor check
3. Prefix U-line residual target ablation
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from prefix_u_line_residual_target import (
    FULL_REPLAY_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    load_known_prefix_anchors,
    run_prefix_u_line_residual_target,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Cache parent:", cfg_get(config, "lineage.cache_parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Active modes:", cfg_get(config, "model.training.active_modes"))
print("Active targets:", cfg_get(config, "model.target_ablation.active_targets"))


## 2. Exp072 full replay cache and prefix anchor check

In [ ]:
cache_path = find_artifact(
    FULL_REPLAY_TRAIN_FEATURES,
    cfg_get(config, "data.exp072_train_feature_cache_local"),
)
print("exp072 full replay train cache:", cache_path)
preview = pd.read_csv(cache_path, nrows=5, dtype={"id": str, "well": str})
print("Columns:", len(preview.columns))
display(preview[[c for c in ["id", "well", "target", "last_known_tvt", "z", "md_since"] if c in preview.columns]])

anchors = load_known_prefix_anchors(paths.train_data_dir, preview["well"].astype(str).unique().tolist())
display(anchors.head())


## 3. Prefix U-line residual target ablation

In [ ]:
prefix_line_config = cfg_get(config, "model.target_ablation.prefix_line", {})
summary = run_prefix_u_line_residual_target(
    output_dir=paths.artifacts_dir,
    train_dir=paths.train_data_dir,
    cache_path=cfg_get(config, "data.exp072_train_feature_cache_local"),
    target_names=cfg_get(config, "model.target_ablation.active_targets", []),
    modes=cfg_get(config, "model.training.modes", {}),
    active_modes=cfg_get(config, "model.training.active_modes", []),
    n_splits=int(cfg_get(config, "validation.n_folds", 5)),
    fast=bool(cfg_get(config, "audit.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_rows=cfg_get(config, "model.training.max_rows"),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    max_lgb_models=cfg_get(config, "model.training.max_lgb_models"),
    prefix_line_alphas=prefix_line_config.get("alphas", [1.0, 0.5]),
    prefix_line_min_rows=int(prefix_line_config.get("min_rows", 8)),
    prefix_line_min_md_span=float(prefix_line_config.get("min_md_span", 25.0)),
    prefix_line_robust_iterations=int(prefix_line_config.get("robust_iterations", 3)),
    save_models=bool(cfg_get(config, "model.training.save_models", True)),
    save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_by_well.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_bucket_metrics.csv")
target_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_target_summary.csv")
manifest_path = paths.artifacts_dir / f"{OUTPUT_PREFIX}_lgb_models" / "manifest.json"

pooled = metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt")
display(pooled)
display(target_summary)
display(bucket_metrics.head(40))
display(by_well.head(30))
print("Model manifest:", manifest_path, "exists=", manifest_path.exists())
